In [1]:
pip install folium

In [2]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
from sklearn.cluster import KMeans

In [3]:
data = pd.DataFrame({
    'latitude': [28.6139, 28.6200, 28.6350, 28.6400, 28.6500, 28.6700],
    'longitude': [77.2090, 77.2100, 77.2200, 77.2300, 77.2400, 77.3000],
    'needy_population': [200, 150, 400, 300, 500, 250]
})
X = data[['latitude', 'longitude']].values
weights = data['needy_population'].values

In [4]:
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans.fit(X, sample_weight=weights)
data['cluster'] = kmeans.labels_

In [5]:
map_center = [data['latitude'].mean(), data['longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=12,title='map')

In [6]:
heat_data = [[row['latitude'], row['longitude'], row['needy_population']] for index, row in data.iterrows()]
HeatMap(heat_data, radius=25).add_to(m)

In [7]:
# Sum needy population for each cluster
cluster_summary = data.groupby('cluster')['needy_population'].sum()
# Convert to dict for easy lookup
cluster_needy = cluster_summary.to_dict()

In [8]:
# Sorting clusters by total needy population (highest first)

sorted_clusters = sorted(cluster_needy.items(), key=lambda x: x[1], reverse=True)
# Assigning colors: most needy = red,green,blue,orange.
cluster_colors = {}
colors_list = ['red', 'green', 'blue', 'orange']
for i, (cluster, _) in enumerate(sorted_clusters):
    cluster_colors[cluster] = colors_list[i]

In [9]:
colors = ['blue', 'green', 'purple', 'orange']

for i, row in data.iterrows():
    cluster_idx = int(row['cluster'])  # <-- casting to int
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=7,
        color=colors[cluster_idx],
        fill=True,
        fill_color=colors[cluster_idx],
        fill_opacity=0.7,
        popup=f"Needy: {row['needy_population']} | Cluster: {cluster_idx}"
    ).add_to(m)

In [10]:
m